In [24]:
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import sklearn

In [25]:
import pandas as pd

df=pd.read_csv('../datasets/bangalore_house.csv')
df.head(5)

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [26]:
df=df.drop(columns=['availability','location','society'])

In [27]:
#function to convert size into numerical column
def extract_bhk(x):
    try:
        return int(str(x).split(' ')[0])
    except:
        return None

In [28]:
df['size_bhk']=df['size'].apply(extract_bhk)
df=df.drop(columns=['size'])
df.head(5)

,area_type,total_sqft,bath,balcony,price,size_bhk
0,Super built-up Area,1056,2.0,1.0,39.07,2.0
1,Plot Area,2600,5.0,3.0,120.00,4.0
2,Built-up Area,1440,2.0,3.0,62.00,3.0
3,Super built-up Area,1521,3.0,1.0,95.00,3.0
4,Super built-up Area,1200,2.0,1.0,51.00,2.0


In [29]:
df=df.fillna(df.mode().iloc[0])

In [30]:
df.isnull().sum()

area_type     0
total_sqft    0
bath          0
balcony       0
price         0
size_bhk      0
dtype: int64

In [31]:
#one hot encode
df = pd.get_dummies(df, columns=['area_type'], drop_first=True)

In [32]:
df.head(10)

,total_sqft,bath,balcony,price,size_bhk,area_type_Carpet Area,area_type_Plot Area,area_type_Super built-up Area
0,1056,2.0,1.0,39.07,2.0,False,False,True
1,2600,5.0,3.0,120.00,4.0,False,True,False
2,1440,2.0,3.0,62.00,3.0,False,False,False
3,1521,3.0,1.0,95.00,3.0,False,False,True
4,1200,2.0,1.0,51.00,2.0,False,False,True
5,1170,2.0,1.0,38.00,2.0,False,False,True
6,2732,4.0,2.0,204.00,4.0,False,False,True
7,3300,4.0,2.0,600.00,4.0,False,False,True
8,1310,3.0,1.0,63.25,3.0,False,False,True
9,1020,6.0,2.0,370.00,6.0,False,True,False


In [33]:
def convert_sqft(x):
    if isinstance(x, str):
        if '-' in x:
            a, b = x.split('-')
            return (float(a) + float(b)) / 2
        try:
            return float(x)
        except:
            return None
    return x

In [34]:
df['total_sqft'] = df['total_sqft'].apply(convert_sqft)

In [35]:
df['total_sqft'] = df['total_sqft'].fillna(df['total_sqft'].mean())

In [36]:
#feature scaling 
#numerical columns -- total_sqft ,bath,balcony,size
from sklearn.preprocessing import MinMaxScaler
MS=MinMaxScaler()
columns=['total_sqft','bath','balcony','size_bhk']
df[columns]=MS.fit_transform(df[columns])

In [37]:
df.head(5)

,total_sqft,bath,balcony,price,size_bhk,area_type_Carpet Area,area_type_Plot Area,area_type_Super built-up Area
0,0.020183,0.025641,0.333333,39.07,0.023810,False,False,True
1,0.049722,0.102564,1.000000,120.00,0.071429,False,True,False
2,0.027530,0.025641,1.000000,62.00,0.047619,False,False,False
3,0.029079,0.051282,0.333333,95.00,0.047619,False,False,True
4,0.022938,0.025641,0.333333,51.00,0.023810,False,False,True


In [38]:
from sklearn.model_selection import train_test_split

In [39]:
X = df.drop('price', axis=1)   # features
y = df['price']                # target
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=5)

In [40]:
df.head(5)

,total_sqft,bath,balcony,price,size_bhk,area_type_Carpet Area,area_type_Plot Area,area_type_Super built-up Area
0,0.020183,0.025641,0.333333,39.07,0.023810,False,False,True
1,0.049722,0.102564,1.000000,120.00,0.071429,False,True,False
2,0.027530,0.025641,1.000000,62.00,0.047619,False,False,False
3,0.029079,0.051282,0.333333,95.00,0.047619,False,False,True
4,0.022938,0.025641,0.333333,51.00,0.023810,False,False,True


In [41]:
from sklearn.linear_model import LinearRegression
model=LinearRegression()
model.fit(X_train,y_train)
y_pred=model.predict(X_test)

In [42]:
from sklearn.metrics import mean_squared_error,r2_score

In [43]:
print("Mean Squared Error",mean_squared_error(y_test,y_pred))

Mean Squared Error 11614.716465146912


In [44]:
print("R2 Score:", r2_score(y_test, y_pred))

R2 Score: 0.48541825550758266


In [45]:
# Step 1: create raw data
new_house = pd.DataFrame([{
    'total_sqft': 1500,
    'bath': 2,
    'balcony': 1,
    'size_bhk': 2,
    'area_type_Built-up  Area': 1
}])

# Step 2: fix columns FIRST
new_house = new_house.reindex(columns=X.columns, fill_value=0)

# Step 3: scale AFTER fixing columns
cols = ['total_sqft', 'bath', 'balcony', 'size_bhk']
new_house[cols] = MS.transform(new_house[cols])

# Step 4: predict
prediction = model.predict(new_house)
print(prediction)

[80.33037359]


In [46]:
import joblib
# Save trained ML model
joblib.dump(model, 'house_price_model.pkl')

# Save scaler
joblib.dump(MS, 'scaler.pkl')

print("Files saved successfully")

Files saved successfully
